# RQ1: Video-quality metrics and human enjoyment

**Research question:** How strongly are scores on selected VBench and VBench-2.0 dimensions associated with human viewers' enjoyment ratings of AI-generated drama clips?

Primary analysis: one Spearman correlation per prespecified dimension against mean human enjoyment per clip, with percentile bootstrap 95% confidence intervals, two-sided Monte Carlo permutation tests, and Holm correction across all selected dimensions.

**Data inputs (produced outside this notebook):**

- `data/metric_scores.csv` — from `notebooks/vbench_score_clips.ipynb`. On Colab that notebook downloads `rq1_vbench_csvs.zip`; unzip it into this `data/` folder first.
- `data/human_ratings.csv` — hand-built from Google Docs (`video_id`, `rater_id`, `enjoyment`)

This notebook does not run VBench and does not invent enjoyment ratings. If either CSV is missing, stop after the load cell.

## Scope and assumptions
- Unit of analysis: clip, not individual rating.
- Mean enjoyment assumes approximately equally spaced scale points. A median sensitivity analysis is included.
- Inference assumes independent clips. Repeated `source_id` values stop confirmatory inference (replace provisional `unlabeled_pilot` with real drama IDs first).
- Clip bootstrap intervals condition on the observed viewer panel.
- Correlations are not causal effects. A nonsignificant result is not evidence of no association.

`DEMO_MODE = True` builds a labeled synthetic panel so the statistical code can be inspected without study files. Those prints are not findings.


In [ ]:
from pathlib import Path
import json, os, platform, warnings
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr, rankdata
import statsmodels
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from IPython.display import display

DEMO_MODE = False

REPO_ROOT = Path("..").resolve()
if not (REPO_ROOT / "data").exists() and (Path.cwd() / "data").exists():
    REPO_ROOT = Path.cwd().resolve()
DATA_DIR = Path(os.environ.get("RQ1_DATA_DIR", REPO_ROOT / "data")).resolve()
OUT_DIR = Path(os.environ.get(
    "RQ1_OUT_DIR",
    REPO_ROOT / ("rq1_outputs_demo" if DEMO_MODE else "rq1_outputs"),
)).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

DIMENSIONS = [
    "vbench_subject_consistency",
    "vbench_background_consistency",
    "vbench_motion_smoothness",
    "vbench_dynamic_degree",
    "vbench_aesthetic_quality",
    "vbench_imaging_quality",
    "vbench2_human_anatomy",
    "vbench2_human_identity",
    "vbench2_human_clothes",
]
SCALE_MIN, SCALE_MAX = 1, 5
MIN_RATERS_PER_CLIP = 3
ALPHA = 0.05
N_BOOTSTRAP = 2000
N_PERMUTATIONS = 4999
SEED = 42

ANALYSIS_READY = False
print("DATA_DIR =", DATA_DIR)
print("OUT_DIR =", OUT_DIR)


## 1. Load data

Research mode reads `data/metric_scores.csv` and `data/human_ratings.csv`. Demo mode uses synthetic rows only.


In [ ]:
rng = np.random.default_rng(SEED)
pd.DataFrame(columns=["video_id", "source_id"] + DIMENSIONS).to_csv(
    OUT_DIR / "metric_scores_template.csv", index=False
)
pd.DataFrame(columns=["video_id", "rater_id", "enjoyment"]).to_csv(
    OUT_DIR / "human_ratings_template.csv", index=False
)

metrics_path = DATA_DIR / "metric_scores.csv"
ratings_path = DATA_DIR / "human_ratings.csv"

if DEMO_MODE:
    n = 40
    ids = [f"clip_{i:03d}" for i in range(n)]
    metrics = pd.DataFrame({"video_id": ids, "source_id": [f"drama_{i:03d}" for i in range(n)]})
    for dimension in DIMENSIONS:
        metrics[dimension] = rng.uniform(0, 1, n)
    latent = (
        2
        + 1.5 * metrics[DIMENSIONS[0]].to_numpy()
        + 0.6 * metrics[DIMENSIONS[1]].to_numpy()
    )
    rows = []
    for j in range(12):
        bias = rng.normal(0, 0.3)
        for i, video_id in enumerate(ids):
            rating = int(np.clip(
                np.rint(latent[i] + bias + rng.normal(0, 0.8)),
                SCALE_MIN, SCALE_MAX,
            ))
            rows.append((video_id, f"viewer_{j:03d}", rating))
    ratings = pd.DataFrame(rows, columns=["video_id", "rater_id", "enjoyment"])
    ANALYSIS_READY = True
    print("SYNTHETIC DEMO - not research results")
else:
    if not metrics_path.is_file():
        raise FileNotFoundError(
            f"Missing {metrics_path}. Run notebooks/vbench_score_clips.ipynb first."
        )
    metrics = pd.read_csv(metrics_path, dtype={"video_id": str, "source_id": str})
    if not ratings_path.is_file():
        ANALYSIS_READY = False
        ratings = pd.DataFrame(columns=["video_id", "rater_id", "enjoyment"])
        print(
            f"Missing {ratings_path}. "
            "Create it from Google Docs with columns video_id,rater_id,enjoyment "
            "(integers 1-5). Association cells will not run until that file exists."
        )
        display(metrics.head())
    else:
        ratings = pd.read_csv(ratings_path, dtype={"video_id": str, "rater_id": str})
        ANALYSIS_READY = True
        display(metrics.head(), ratings.head())


In [ ]:
assert len(DIMENSIONS) > 0 and len(set(DIMENSIONS)) == len(DIMENSIONS)
assert {"video_id", *DIMENSIONS}.issubset(metrics.columns), "Missing metric columns"

if not ANALYSIS_READY:
    analysis = pd.DataFrame()
    print("Aggregation skipped until human_ratings.csv exists.")
else:
    assert {"video_id", "rater_id", "enjoyment"}.issubset(ratings.columns), "Missing rating columns"
    assert not metrics["video_id"].isna().any() and not ratings[["video_id", "rater_id"]].isna().any().any()
    assert not metrics["video_id"].duplicated().any(), "Metric table must have one row per clip"
    assert not ratings.duplicated(["video_id", "rater_id"]).any(), "Resolve duplicate viewer-clip records"
    for col in DIMENSIONS:
        metrics[col] = pd.to_numeric(metrics[col], errors="raise")
    ratings["enjoyment"] = pd.to_numeric(ratings["enjoyment"], errors="raise")
    assert not np.isinf(metrics[DIMENSIONS].to_numpy(dtype=float)).any()
    assert not np.isinf(ratings["enjoyment"].to_numpy(dtype=float)).any()
    observed = ratings["enjoyment"].dropna()
    assert observed.between(SCALE_MIN, SCALE_MAX).all(), "Ratings outside configured scale"
    unknown = set(ratings["video_id"]) - set(metrics["video_id"])
    assert not unknown, f"Ratings without metric-table records: {unknown}"
    print("Missing enjoyment rows excluded:", int(ratings["enjoyment"].isna().sum()))
    valid_ratings = ratings.dropna(subset=["enjoyment"]).copy()
    summary = valid_ratings.groupby("video_id")["enjoyment"].agg(
        human_mean="mean", human_median="median", n_raters="count", human_sd="std"
    ).reset_index()
    clips = metrics.merge(summary, on="video_id", how="left", validate="one_to_one")
    clips["included"] = clips["n_raters"].fillna(0).ge(MIN_RATERS_PER_CLIP)
    clips.to_csv(OUT_DIR / "clip_summary_and_exclusions.csv", index=False)
    print("Clips excluded for insufficient ratings:", int((~clips["included"]).sum()))
    analysis = clips.loc[clips["included"]].copy()
    assert len(analysis) >= 4, "At least four eligible clips required; substantially more are recommended"
    if "source_id" in analysis:
        assert not analysis["source_id"].isna().any(), "Fill source IDs or remove the optional column"
        assert not analysis["source_id"].duplicated().any(), (
            "Repeated drama/source IDs: independent-clip inference is inappropriate. "
            "Replace provisional unlabeled_pilot values with real drama IDs "
            "(one clip per independent source), or use a cluster-aware design."
        )
    else:
        warnings.warn("No source_id supplied: verify clip independence manually.")
    display(analysis[["n_raters", "human_mean", "human_sd"]].describe())


## 2. Spearman correlations and uncertainty

Permutation tests shuffle clip-level enjoyment relative to metric scores. The Monte Carlo p-value uses the absolute correlation and a plus-one correction. Bootstrap intervals resample paired clips and recalculate ranks; constant resamples are discarded and counted. These are marginal, not simultaneous, 95% intervals. Missing metric values are handled separately per dimension, with sample sizes reported. Holm correction includes all prespecified dimensions; untestable dimensions enter the correction conservatively as p = 1 and retain missing reported results.

SciPy recommends considering permutation tests rather than relying on the asymptotic Spearman p-value for small samples: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html

Holm correction documentation: https://www.statsmodels.org/devel/generated/statsmodels.stats.multitest.multipletests.html


In [ ]:
def association(x, y, seed):
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x, float), np.asarray(y, float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    result = dict(
        n_clips=len(x), rho=np.nan, ci_low=np.nan, ci_high=np.nan,
        p_permutation=np.nan, valid_bootstraps=0, status="untestable",
    )
    if len(x) < 4 or len(np.unique(x)) < 2 or len(np.unique(y)) < 2:
        return result
    rx, ry = rankdata(x), rankdata(y)
    rx, ry = rx - rx.mean(), ry - ry.mean()
    denominator = np.linalg.norm(rx) * np.linalg.norm(ry)
    rho = float(rx @ ry / denominator)
    extreme = 0
    for _ in range(N_PERMUTATIONS):
        perm_rho = rx @ rng.permutation(ry) / denominator
        extreme += abs(perm_rho) >= abs(rho) - 1e-12
    boot = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, len(x), len(x))
        if np.unique(x[idx]).size > 1 and np.unique(y[idx]).size > 1:
            boot.append(float(spearmanr(x[idx], y[idx]).statistic))
    low, high = (
        np.quantile(boot, [0.025, 0.975]) if len(boot) >= 100 else (np.nan, np.nan)
    )
    result.update(
        rho=rho, ci_low=low, ci_high=high,
        p_permutation=(extreme + 1) / (N_PERMUTATIONS + 1),
        valid_bootstraps=len(boot), status="tested",
    )
    return result

if ANALYSIS_READY:
    results = pd.DataFrame([
        dict(dimension=d, **association(analysis[d], analysis["human_mean"], SEED + i))
        for i, d in enumerate(DIMENSIONS)
    ])
    testable = results["status"].eq("tested")
    reject, adjusted, _, _ = multipletests(
        results["p_permutation"].fillna(1), alpha=ALPHA, method="holm"
    )
    results["p_holm"] = np.where(testable, adjusted, np.nan)
    results["reject_holm"] = testable & reject
    results["abs_rho"] = results["rho"].abs()
    results = results.sort_values("abs_rho", ascending=False, na_position="last")
    results.to_csv(OUT_DIR / "rq1_spearman_results.csv", index=False)
    display(results.drop(columns="abs_rho").round(4))
else:
    results = pd.DataFrame()
    print("Spearman cells skipped: waiting for data/human_ratings.csv")


## 3. Plots and median sensitivity analysis

Sensitivity correlations are descriptive checks, not a second set of confirmatory tests.


In [ ]:
if not ANALYSIS_READY:
    print("Plots skipped: waiting for data/human_ratings.csv")
else:
    plot_data = results.dropna(subset=["rho"]).iloc[::-1]
    if len(plot_data):
        fig, ax = plt.subplots(figsize=(9, max(3, len(plot_data) * 0.5)))
        for i, row in enumerate(plot_data.itertuples()):
            color = "tab:blue" if row.reject_holm else "gray"
            ax.plot([row.ci_low, row.ci_high], [i, i], color=color)
            ax.scatter(row.rho, i, color=color)
        ax.set_yticks(range(len(plot_data)), plot_data["dimension"])
        ax.axvline(0, color="black", lw=0.8)
        ax.set(
            xlim=(-1.05, 1.05),
            xlabel="Spearman rho (marginal bootstrap 95% CI)",
            title=("DEMO: " if DEMO_MODE else "") + "Metric-enjoyment associations",
        )
        fig.tight_layout()
        fig.savefig(OUT_DIR / "rq1_correlations.png", dpi=180, bbox_inches="tight")
        plt.show()

    sensitivity = []
    for d in DIMENSIONS:
        subset = analysis[[d, "human_median"]].dropna()
        rho = (
            spearmanr(subset[d], subset["human_median"]).statistic
            if (
                len(subset) >= 4
                and subset[d].nunique() > 1
                and subset["human_median"].nunique() > 1
            )
            else np.nan
        )
        sensitivity.append({"dimension": d, "n_clips": len(subset), "rho_human_median": rho})
    sensitivity = pd.DataFrame(sensitivity)
    sensitivity.to_csv(OUT_DIR / "median_sensitivity.csv", index=False)
    display(sensitivity)

    fig, axes = plt.subplots(len(DIMENSIONS), 1, figsize=(7, 3 * len(DIMENSIONS)), squeeze=False)
    for ax, d in zip(axes.flat, DIMENSIONS):
        ax.scatter(analysis[d], analysis["human_mean"], alpha=0.7)
        ax.set(xlabel=d, ylabel="Mean human enjoyment", ylim=(SCALE_MIN - 0.1, SCALE_MAX + 0.1))
    fig.suptitle("SYNTHETIC DEMO" if DEMO_MODE else "Clip-level observations")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "rq1_scatterplots.png", dpi=180, bbox_inches="tight")
    plt.show()


In [ ]:
metadata = {
    "demo_mode": DEMO_MODE,
    "analysis_ready": ANALYSIS_READY,
    "dimensions": DIMENSIONS,
    "scale": [SCALE_MIN, SCALE_MAX],
    "minimum_raters": MIN_RATERS_PER_CLIP,
    "alpha": ALPHA,
    "bootstrap_resamples": N_BOOTSTRAP,
    "permutations": N_PERMUTATIONS,
    "seed": SEED,
    "eligible_clips": int(len(analysis)) if ANALYSIS_READY else 0,
    "data_dir": str(DATA_DIR),
    "metrics_csv": str(metrics_path if not DEMO_MODE else "synthetic"),
    "ratings_csv": str(ratings_path if not DEMO_MODE else "synthetic"),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
}
(OUT_DIR / "analysis_settings.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

if not ANALYSIS_READY:
    report = (
        "RQ1 association waiting for data/human_ratings.csv.\n"
        "metric_scores.csv is loaded from the VBench scoring notebook when present.\n"
        "Add Google Docs ratings manually, then rerun this notebook."
    )
else:
    lines = [
        "SYNTHETIC DEMO - NOT STUDY FINDINGS" if DEMO_MODE else "RQ1 analysis summary",
        f"Eligible clips: {len(analysis)}; mean enjoyment aggregated per clip.",
        "Two-sided clip permutation p-values; Holm correction across selected dimensions.",
        "Intervals are marginal paired-clip bootstrap intervals, conditional on the observed viewer panel.",
    ]
    for row in results.itertuples():
        if row.status == "tested":
            lines.append(
                f"{row.dimension}: rho={row.rho:.3f}, 95% CI [{row.ci_low:.3f}, {row.ci_high:.3f}], "
                f"Holm p={row.p_holm:.4f}, n={row.n_clips}."
            )
        else:
            lines.append(f"{row.dimension}: untestable (insufficient data or no variation).")
    lines += [
        "Associations are not causal effects. Nonsignificance is not evidence of no association.",
        "Largest absolute sample correlation does not prove a dimension is stronger than all others.",
    ]
    report = "\n".join(lines)

(OUT_DIR / "rq1_summary.txt").write_text(report, encoding="utf-8")
print(report)


## Reporting checklist

Report clip and viewer counts, source sampling, scale anchors, metric versions/configurations, chosen dimensions and rationale, missingness/exclusions, per-dimension sample sizes, rho and confidence intervals, Holm-adjusted p-values, and limitations of clip independence and viewer sampling.

Example sentence (replace placeholders; never copy demo estimates):

> Across N eligible clips, dimension X was positively associated with mean viewer enjoyment (Spearman rho = R, marginal bootstrap 95% CI [L, U], Holm-adjusted permutation p = P). This association does not establish causation or an independent contribution beyond other dimensions.

Scoring: `notebooks/vbench_score_clips.ipynb`. Ratings: manual `data/human_ratings.csv` from Google Docs.
